# Visual Guardian V2 — Fall Dataset Verification
**Accelerator:** CPU (no GPU needed)  
**Input:** `payutch/fall-video-dataset`  
**Output:** `fall_verification_manifest.csv` → save as Kaggle dataset → input for preprocessing notebook

In [ ]:
import re
import cv2
import numpy as np
import pandas as pd
from pathlib import Path

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
MIN_FRAMES   = 32
MIN_DURATION = 1.0
VIDEO_EXTS   = {".mp4", ".avi", ".mov", ".mkv", ".wmv"}

# Auto-detect dataset root: find the folder containing Fall/ and No_Fall/
DATASET_ROOT = None
for d in Path("/kaggle/input").rglob("Fall"):
    candidate = d.parent
    if (candidate / "No_Fall").exists():
        DATASET_ROOT = candidate
        break

if DATASET_ROOT is None:
    raise RuntimeError("Could not find Fall/ and No_Fall/ folders. Attach payutch/fall-video-dataset.")

FALL_DIR  = DATASET_ROOT / "Fall"    / "Raw_Video"
NFALL_DIR = DATASET_ROOT / "No_Fall" / "Raw_Video"
print(f"Dataset root : {DATASET_ROOT}")
print(f"Fall videos  : {sum(1 for f in FALL_DIR.iterdir() if f.suffix.lower() in VIDEO_EXTS)}")
print(f"No-fall videos: {sum(1 for f in NFALL_DIR.iterdir() if f.suffix.lower() in VIDEO_EXTS)}")

In [ ]:
# ── Subject ID Extraction ─────────────────────────────────────────────────────
def extract_subject_id(filename: str) -> str | None:
    stem = Path(filename).stem

    # Pattern 1 — source_person_clip e.g. S_N_233, C_M_107
    m = re.match(r'^([A-Za-z])_([A-Za-z])_', stem)
    if m:
        return f"{m.group(1).upper()}_{m.group(2).upper()}"

    # Pattern 2 — timestamp e.g. 20240918190020 (group by day)
    m2 = re.match(r'^(\d{8})', stem)
    if m2:
        return f"DATE_{m2.group(1)}"

    return None

In [ ]:
# ── Collect All Video Files ───────────────────────────────────────────────────
records = []
for label, folder in [("fall", FALL_DIR), ("no_fall", NFALL_DIR)]:
    clips = [f for f in folder.iterdir() if f.is_file() and f.suffix.lower() in VIDEO_EXTS]
    print(f"{label}: {len(clips)} clips")
    for path in clips:
        records.append({
            "path":       str(path),
            "filename":   path.name,
            "label":      label,
            "subject_id": extract_subject_id(path.name),
            "size_bytes": path.stat().st_size,
        })

df = pd.DataFrame(records)
print(f"\nTotal: {len(df)} clips")
df["label"].value_counts()

In [ ]:
# ── Subject ID Analysis ───────────────────────────────────────────────────────
missing = df["subject_id"].isna().sum()
print(f"Subject IDs resolved : {df['subject_id'].notna().sum()} / {len(df)}")
print(f"Missing subject IDs  : {missing}")
if missing > 0:
    print("\nExamples of unresolved filenames:")
    for f in df[df["subject_id"].isna()]["filename"].head(5).values:
        print(f"  {f}")
print(f"\nSubject group distribution:")
print(df["subject_id"].value_counts().to_string())
print(f"\nUnique groups: {df['subject_id'].nunique()}")

In [ ]:
# ── Fast Metadata Validation (metadata-only, no frame decoding) ───────────────
frame_counts, fps_list, widths, heights, readable = [], [], [], [], []

for i, row in df.iterrows():
    cap = cv2.VideoCapture(row["path"])
    if not cap.isOpened():
        frame_counts.append(0); fps_list.append(0.0)
        widths.append(0); heights.append(0); readable.append(False)
        cap.release(); continue

    fc  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Only if metadata says 0 — do a quick single-frame read
    is_ok = (fc > 0) or cap.read()[0]
    cap.release()

    frame_counts.append(fc); fps_list.append(round(fps, 2))
    widths.append(w); heights.append(h); readable.append(is_ok)

    if (i + 1) % 500 == 0:
        print(f"  {i+1}/{len(df)} done...")

df["frame_count"] = frame_counts
df["fps"]         = fps_list
df["width"]       = widths
df["height"]      = heights
df["readable"]    = readable
df["duration_s"]  = (df["frame_count"] / df["fps"].replace(0, np.nan)).round(2)
print("Done.")

In [ ]:
# ── Quality Report ────────────────────────────────────────────────────────────
valid_df = df[
    df["readable"] &
    (df["frame_count"] >= MIN_FRAMES) &
    (df["duration_s"]  >= MIN_DURATION)
].copy()

print(f"Total clips          : {len(df)}")
print(f"Unreadable/corrupt   : {(~df['readable']).sum()}")
print(f"Below {MIN_FRAMES} frames       : {(df['readable'] & (df['frame_count'] < MIN_FRAMES)).sum()}")
print(f"Below {MIN_DURATION}s duration  : {(df['readable'] & (df['duration_s'] < MIN_DURATION)).sum()}")
print(f"Valid clips          : {len(valid_df)}")
print(f"\nValid by label:")
print(valid_df["label"].value_counts().to_string())
print(f"\nFrame count stats:")
print(valid_df["frame_count"].describe().round(1).to_string())
print(f"\nDuration stats (seconds):")
print(valid_df["duration_s"].describe().round(2).to_string())
print(f"\nFPS distribution:")
print(valid_df["fps"].value_counts().head(8).to_string())
valid_df["resolution"] = valid_df["width"].astype(str) + "x" + valid_df["height"].astype(str)
print(f"\nResolution distribution:")
print(valid_df["resolution"].value_counts().head(8).to_string())

In [ ]:
# ── Split Strategy + Go/No-Go + Save ─────────────────────────────────────────
n_with_id = valid_df["subject_id"].notna().sum()
pct       = 100 * n_with_id / max(len(valid_df), 1)
print(f"Clips with group ID: {n_with_id} / {len(valid_df)} ({pct:.1f}%)")
strategy  = "SUBJECT-LEVEL" if pct >= 80 else "CLIP-LEVEL RANDOM"
print(f"Split strategy: {strategy}")

fall_ct   = (valid_df["label"] == "fall").sum()
nofall_ct = (valid_df["label"] == "no_fall").sum()
ratio     = max(fall_ct, nofall_ct) / max(min(fall_ct, nofall_ct), 1)
print(f"\nClass balance: fall={fall_ct}, no_fall={nofall_ct}, ratio={ratio:.2f}x")
if ratio > 2.5:
    print("  WARN: Significant imbalance — use class weights in training")
else:
    print("  OK: Balanced enough — no class weights needed")

out = "/kaggle/working/fall_verification_manifest.csv"
df.to_csv(out, index=False)
print(f"\nVerdict: GO — {len(valid_df)} valid clips")
print(f"Manifest saved: {out}")
print("Save /kaggle/working as a Kaggle dataset, then attach it to the preprocessing notebook.")